# 🔍 Exploración y Modelado de Detección de Fraude (`card_transaction.v1.csv`)

Este notebook está configurado para utilizar el kernel **Python 3.12 (IA & Data - iaupeu)**.
Aprovecharemos **Polars** para lectura ultra-rápida (1 millón de registros), análisis exploratorio con Seaborn/Plotly y modelos de clasificación con LightGBM/XGBoost.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Carga Rápida de Datos con Polars
`card_transaction.v1.csv` contiene 1 millón de registros (~150 MB). Polars lo procesa en milisegundos usando multithreading nativo en Rust.

In [6]:
import polars as pl
from pathlib import Path

DATASET_PATH = "data/DATA IBM/card_transaction.v1.csv"

# Lectura con Polars
df_pl = pl.read_csv(DATASET_PATH)
print(f"Dimensiones del dataset: {df_pl.shape[0]:,} filas x {df_pl.shape[1]} columnas")
df_pl.head(5)

Dimensiones del dataset: 24,386,900 filas x 15 columnas


User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
i64,i64,i64,i64,i64,str,str,str,i64,str,str,f64,i64,str,str
0,0,2002,9,1,"""06:21""","""$134.09""","""Swipe Transaction""",3527213246127876953,"""La Verne""","""CA""",91750.0,5300,null,"""No"""
0,0,2002,9,1,"""06:42""","""$38.48""","""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""
0,0,2002,9,2,"""06:22""","""$120.34""","""Swipe Transaction""",-727612092139916043,"""Monterey Park""","""CA""",91754.0,5411,null,"""No"""
0,0,2002,9,2,"""17:45""","""$128.95""","""Swipe Transaction""",3414527459579106770,"""Monterey Park""","""CA""",91754.0,5651,null,"""No"""
0,0,2002,9,3,"""06:23""","""$104.71""","""Swipe Transaction""",5817218446178736267,"""La Verne""","""CA""",91750.0,5912,null,"""No"""


## 2. Distribución de la Variable Objetivo (`is_fraud`)
Revisamos el balance entre transacciones legítimas (0) y fraudulentas (1).

In [3]:
fraud_counts = df_pl["is_fraud"].value_counts().to_pandas()
fraud_counts["percentage"] = (fraud_counts["count"] / df_pl.shape[0]) * 100
print(fraud_counts)

# Tipo de fraude
print("\nTipos de fraude:")
print(df_pl["fraud_type"].value_counts().to_pandas())

   is_fraud   count  percentage
0         0  944745     94.4745
1         1   55255      5.5255

Tipos de fraude:
           fraud_type   count
0        Card Cloning    9200
1      Friendly Fraud    9187
2            Phishing    9252
3                None  944745
4  Synthetic Identity    9214
5    Account Takeover    9232
6      Identity Theft    9170


## 3. Integración con Microsoft Data Wrangler
> **Tip:** Convierte una muestra o el dataset a Pandas y haz clic en el icono de **Data Wrangler** en la pestaña de variables de Jupyter para explorar correlaciones y distribuciones visuales de forma interactiva.

In [ ]:
# Muestra de trabajo para Pandas / Data Wrangler
df_sample = df_pl.sample(n=100000, seed=42).to_pandas()
df_sample.describe()